<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Boosting

_Author:_ Tim Book

---

### Learning Objectives

- Understand the reaons to perform bagging vs boosting.
- Understand how boosting is an ensemble method.
- Learn the pros and cons to using boosting models.
- Learn the effect of boosting on the bias-variance trade-off.
- Learn the math and procedure for AdaBoost, the "classic" boosting model.
- Understand the differences between AdaBoost and gradient-boosting models.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

### Today's Data
We're going to be using the famous spam detection dataset. It's documented in UCI's ML database [here](https://archive.ics.uci.edu/ml/datasets/spambase) and is used frequently throughout the Holy Text (ESL) for tree-based models.

![](images/spam.png)

In [2]:
names_raw = open('data/spam-colnames.txt', 'r').readlines()
names = [line.split(':')[0] for line in names_raw[33:]] + ['is_spam']
spam = pd.read_csv('data/spam.csv', sep='\s+', names=names)
spam.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [3]:
# Assumption: Data are very clean! No missings, categorical features, etc.
# Let's move a long here for the sake of time.

In [4]:
X = spam.drop(columns=['is_spam'])
y = spam['is_spam']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1234)

### The AdaBoost model
`sklearn` implements the SAMME.R algorithm, not AdaBoost.M1 like we learned in class, but it's very similar.

[Docs here!](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)

In [6]:
# Let's take a peek at the docs...
AdaBoostClassifier()

AdaBoostClassifier()

In [7]:
# Try max_depth=None vs max_depth=1
tree = DecisionTreeClassifier(max_depth=1, random_state=111)
ada = AdaBoostClassifier(tree, n_estimators=100, learning_rate=1, algorithm='SAMME.R')

In [8]:
ada.fit(X_train, y_train)

AdaBoostClassifier(base_estimator=DecisionTreeClassifier(max_depth=1,
                                                         random_state=111),
                   learning_rate=1, n_estimators=100)

In [9]:
print(ada.score(X_train, y_train))
print(ada.score(X_test, y_test))

0.9666666666666667
0.9357080799304952


### The GradientBoost model
Remember the strategy and impetus behind gradient boosting is similar to AdaBoosting with a few differences. The main one is that you can choose your loss function! The `sklearn` implementation already pre-incorporates the tree model, so we get to skip a step there.

[Docs here!](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html#sklearn.ensemble.GradientBoostingClassifier)

In [10]:
# Instantiate and fit
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)

GradientBoostingClassifier()

In [11]:
# Easy peasy - didn't even mess with tuning params!
print(gb.score(X_train, y_train))
print(gb.score(X_test, y_test))

0.9672463768115942
0.9391833188531712


### Let's GridSearch!

In [12]:
pgrid = {
    'learning_rate': [0.1, 1, 10],
    'n_estimators': [10, 100],
    'max_depth': [None, 1, 2, 3]
}

gb = GradientBoostingClassifier(random_state=222)
gs = GridSearchCV(gb, pgrid, cv=5, n_jobs=6)

In [13]:
%%time
gs.fit(X_train, y_train)

CPU times: user 983 ms, sys: 77.6 ms, total: 1.06 s
Wall time: 25.5 s


GridSearchCV(cv=5, estimator=GradientBoostingClassifier(random_state=222),
             n_jobs=6,
             param_grid={'learning_rate': [0.1, 1, 10],
                         'max_depth': [None, 1, 2, 3],
                         'n_estimators': [10, 100]})

In [14]:
print(gs.score(X_train, y_train))
print(gs.score(X_test, y_test))

0.9710144927536232
0.9348392701998263


In [15]:
gs.best_params_

{'learning_rate': 1, 'max_depth': 1, 'n_estimators': 100}

<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Conclusion
---
<br>
<br>

* Boosting is a very powerful technique that is **best used to improve the bias of weak learners.** 
* Boosting reduces bias by improving weak learners, contrast this with bagging with reduces variance by average too-strong (overfit) learners.
* You can mess with the tuning parameters if you'd like, but generally if you train for long enough, the learning rate won't matter.

### AdaBoost vs GradientBoost
- AdaBoost is about reweighting the preceding model's errors in subsequent iterations.
- Gradient boosting is about fitting subsequent models to the residuals of the last model.

### Model Boosting Pros
* Top-tier performance
* You get all the same benefits associated with tree-based models

### Model Boosting Cons
* All hope of interpretation, gone!
* Computational complexity is the highest we've ever seen - can cause models to fit slowly!
* Because of serial nature of algorithm, parallelization is impossible.